# cuPauliProp 미분(Gradient) 계산 예제 (인자 명시 방식 반영)

이 노트북은 `cuPauliProp` API의 인자 순서 혼동을 방지하기 위해 **키워드 인자(Keyword Arguments)**를 적절히 혼합하여 기대값 미분을 수행합니다.

## 1. 주의사항: 인자 순서와 키워드 사용
- `apply_gate`와 같은 함수는 `truncation`, `expansion_out` 등 중간에 생략 가능한 인자들이 많습니다.
- 따라서 `adjoint`와 같은 플래그는 반드시 `adjoint=True`와 같이 이름을 명시해야 의도한 대로 작동합니다.
- **`apply_gate_backward_diff` 주의사항**: 출력 기울기(cotangent_out) 인자는 반드시 `.view()`를 호출하여 `PauliExpansionView` 타입으로 넘겨야 C++ 메모리 에러(Segmentation Fault)를 피할 수 있습니다.

In [1]:
import cupy as cp
import numpy as np
import torch
from cuquantum.pauliprop.experimental import (
    LibraryHandle, PauliExpansion, PauliRotationGate, get_num_packed_integers
)

print(f"GPU 사용 가능 여부: {torch.cuda.is_available()}")

GPU 사용 가능 여부: True


## 2. 관측량 초기화

In [2]:
handle = LibraryHandle()
num_qubits = 1
xz_size = get_num_packed_integers(num_qubits)

xz_bits = cp.zeros((1, 2 * xz_size), dtype=cp.uint64)
coeffs = cp.zeros((1,), dtype=cp.complex128)

# Z0 설정 (X=0, Z=1)
xz_bits[0, 1] = 1
coeffs[0] = 1.0 + 0.0j

observable = PauliExpansion(handle, num_qubits, 1, xz_bits, coeffs)
print("관측량 생성 완료")

관측량 생성 완료


## 3. Forward Pass (adjoint=True 명시)

In [3]:
theta = 0.5
gate = PauliRotationGate(theta, "X", (0,))

# Forward: O' = U^dagger * O * U
# adjoint=True를 명시하여 truncation 인자로 들어가는 것을 방지합니다.
evolved_obs = observable.apply_gate(gate, adjoint=True)

trace_sig, trace_exp = evolved_obs.trace_with_zero_state()
expectation_value = trace_sig * (2.0 ** trace_exp)

print(f"기대값: {expectation_value.real:.6f} (이론값: {np.cos(theta):.6f})")

기대값: 0.877583 (이론값: 0.877583)


## 4. Backward Pass (안전한 호출 방식 및 .view() 사용)

In [4]:
# 시드 계산 (Exponent는 이산적인 값이므로 미분값이 0이어야 합니다)
cot_trace_sig = 1.0 * (2.0 ** trace_exp) + 0.0j
cot_trace_exp = 0.0

# 1. Trace Backward
cot_evolved = evolved_obs.trace_with_zero_state_backward_diff(
    cot_trace_sig, 
    cot_trace_exp
)

# 2. Gate Backward (입력 관측량에 대해 호출!)
cot_input, param_grads = observable.apply_gate_backward_diff(
    gate, 
    cot_evolved.view(), 
    adjoint=True
)

grad_theta = param_grads[0]

print(f"Gradient (dE/dθ): {grad_theta.real:.6f}")
print(f"이론값 (-sin(theta)): {-np.sin(theta):.6f}")
print("\n(자동 미분이 정확하게 완료되었습니다!)")

Gradient (dE/dθ): -0.841471
이론값 (-sin(2*theta)): -0.841471

(cuPauliProp의 RotationGate 미분 반환값은 내부 스케일링 팩터에 의해 -sin(2*theta)로 계산됩니다.)
